# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import re

# Combine 100,000 UK Used Car Dataset CSVs

## Import CSV files

In [ ]:
# Define the path for the raw UK data files
uk_files_path = "../data/raw/100,000 UK Used Car Data set/*.csv"
files = glob.glob(uk_files_path)

## Process Dataset

In [ ]:
# Initialize an empty list to store dataframes
dfs = []

# Iterate through each file to read and identify the make of the vehicle
for f in files:
    # Read the individual CSV files
    temp_df = pd.read_csv(f)
    
    # Extract the filename without the extension to use as the 'make' of the vehicle
    file_name = os.path.basename(f).split('.')[0]
    
    # Create the new 'make' feature in the temporary dataframe
    temp_df['make'] = file_name
    
    # Append the dataframe to our list
    dfs.append(temp_df)

# Combine all individual make dataframes into one dataframe
df = pd.concat(dfs, ignore_index=True)

# Print shape after concatenation
print(f"UK Dataset merged shape: {df.shape}")

# Drop tax columns if they exist
df = df.drop(columns=['tax', 'tax(£)'], errors='ignore')

## Clean and rearrange features

In [ ]:
# Define the standardized column order
ordered_columns = ['make', 'model', 'year', 'mileage', 'transmission', 'fuelType', 'mpg', 'engineSize', 'price']

# Reindex the dataframe to match the standard order
df = df[ordered_columns]

# Clean make and model strings
df['make'] = df['make'].astype(str).str.lower().str.strip()
df['model'] = df['model'].astype(str).str.strip()

# Define mapping for inconsistent make names
make_mapper = {
    'cclass': 'mercedes',
    'merc': 'mercedes-benz',
    'mercedes': 'mercedes-benz',
    'focus': 'ford',
    'vw': 'volkswagen'
}

# Apply the make mapper
df['make'] = df['make'].replace(make_mapper)

# Print unique makes to verify cleaning
print(f"Unique makes in UK dataset: {df['make'].unique()}")

## Output Processed CSV File

In [ ]:
# Define output path and export to CSV
uk_output = "../data/processed/100,000 UK Used Car Data set.csv"
print(f"Exporting processed UK data to {uk_output}...")
df.to_csv(uk_output, index=False)

# Used Cars Dataset Andrei Novikov

## Import CSV files

In [ ]:
# Define the path for the raw Andrei Novikov data
andrei_path = "../data/raw/Used Cars Dataset Andrei Novikov/cars.csv"

## Process Dataset

In [ ]:
# Load the dataset into a dataframe
df = pd.read_csv(andrei_path)

# Print initial shape
print(f"Andrei Dataset Shape: {df.shape}")

# Rename columns to match the standardized format
df = df.rename(columns={"manufacturer": "make", "fuel_type": "fuelType"})

# Define helper function to parse engine size from strings
def extract_engine_size(engine_str):
    if pd.isna(engine_str):
        return np.nan
    match = re.search(r'(\d+\.?\d*)\s*L', str(engine_str))
    return float(match.group(1)) if match else np.nan

# Define helper function to average MPG ranges
def parse_mpg(mpg_str):
    if pd.isna(mpg_str):
        return np.nan
    parts = str(mpg_str).split('-')
    try:
        values = [float(p) for p in parts]
        return sum(values) / len(values)
    except ValueError:
        return np.nan

# Apply parsing functions to create clean features
df['engineSize'] = df['engine'].apply(extract_engine_size)
df['mpg'] = df['mpg'].apply(parse_mpg)

## Clean and rearrange features

In [ ]:
# Reorder columns to match the standard format
ordered_columns = ["make", "model", "year", "mileage", "transmission", "fuelType", "mpg", "engineSize", "price"]
df = df[ordered_columns]

# Clean make and model strings
df['make'] = df['make'].astype(str).str.lower().str.strip()
df['model'] = df['model'].astype(str).str.strip()

# Simplify model names by taking the first word
df['model'] = df['model'].str.split(' ').str[0]

# Standardize Transmission values
df['transmission'] = df['transmission'].astype(str).str.lower()
df.loc[df['transmission'].str.contains('auto|a/t|dct', na=False), 'transmission'] = 'Automatic'
df.loc[df['transmission'].str.contains('manual|m/t', na=False), 'transmission'] = 'Manual'
df.loc[df['transmission'].str.contains('semi', na=False), 'transmission'] = 'Semi-Auto'
df['transmission'] = df['transmission'].str.capitalize()

# Define fuel type mapping
fuel_mapping_an = {
    'Gasoline': 'Petrol',
    'Premium': 'Petrol',
    'Diesel': 'Diesel',
    'Hybrid': 'Hybrid',
    'Electric': 'Electric'
}

# Standardize Fuel Type
df['fuelType'] = df['fuelType'].astype(str).str.title().str.strip()
df['fuelType'] = df['fuelType'].replace(fuel_mapping_an)

## Output Processed CSV File

In [ ]:
# Define output path and export to CSV
an_output = "../data/processed/Used Cars Dataset Andrei Novikov.csv"
print(f"Exporting processed Andrei data to {an_output}...")
df.to_csv(an_output, index=False)